# Notebook 09 — Real-Data Pipeline on GLODAP (Track 1 v0.95)

**The pipeline test.** Notebooks 05–07 ran the verified DarwinDiff methodology on synthetic ground truth. This notebook is the bridge to real ocean observations: loads GLODAPv2.2016b mapped climatology (NOAA NCEI, public, no auth), subsets to a Mid-Atlantic AOI, and runs the DINN per-cell parameter learner against a real-data spatial pattern.

**Honest scope flag (read this first).** The Carroll-6 parameters and the `carroll6` box model in `src/darwindiff/carroll6.py` were designed as a 5-tracer proxy for Darwin BGC. The proxy's tracers (DFe, $P_s$, $P_l$, POC, PIC) do *not* directly correspond to GLODAP's gridded fields (DIC, ALK, NO3, PO4, Si, oxygen). So this notebook cannot recover Carroll's published optima — the data wasn't produced by Carroll's model. What it *does* demonstrate:

1. **The full data pipeline works on real ocean observations.** Download → xarray load → AOI mask → DINN per-cell prediction → forward integration of `carroll6` box model → autograd back through the integration → Adam updates. Every step that worked on synthetic 2-D fields in notebook 07 also works on the real GLODAP grid with land masking.
2. **The loss converges and the recovered Carroll-6 spatial maps are smooth.** Methodological validation that the optimisation is well-behaved on real-data spatial structure, not just synthetic toy data.
3. **The proxy mapping is biologically defensible.** The loss is MSE between **z-scored phyto biomass** ($P_s + P_l$ from the box model at steady state) and **z-scored negative NO$_3$** from GLODAP — high phytoplankton biomass spatially correlates with low surface NO$_3$ because phytoplankton consume nitrate. Z-scoring both sides decouples the loss from absolute magnitude and avoids the pathological behaviour of $1/\text{NO}_3$ at the small/negative values that GLODAP's mapped product can take in oligotrophic regions.

**What this does *not* prove** (deferred to notebook 10+):

- **Carroll-6 recovery against published optima.** GLODAP wasn't produced by Darwin, so the recovered values aren't compared to Carroll's `(0.928, 6e-7, 0.661, 0.431, 0.830, 0.0425)`. The recovered numbers are best-fit-to-this-loss values, not Darwin-equivalent values.
- **Iron-pair identifiability.** GLODAP has no gridded iron field. `alpfe` and `scav_rat` are unconstrained by this loss. Adding GEOTRACES iron sections is the next-step deliverable for those two parameters.

**Headline result.** Pearson correlation between the DINN-driven box-model phyto biomass pattern and the inverse-NO$_3$ pattern (z-scored), over 582 ocean cells in the Mid-Atlantic AOI: **r ≈ 0.69**. The methodology runs end-to-end on real ocean data, the loss converges, and the recovered Carroll-6 maps are smooth functions of SST in physically-plausible ranges.

In [ ]:
import sys
import time
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from darwindiff.carroll6 import (
    PARAM_BOUNDS,
    PARAM_NAMES,
    bounded_params,
    carroll6_step,
)
from darwindiff.networks import DINN
from darwindiff.diagnostics import format_pearson, safe_pearson_r

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(
    f"GPU detected: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}"
)

## 1. Real ocean data: GLODAPv2.2016b mapped climatology

Source: NOAA NCEI, accession `0162565`. Fully public, no authentication. Downloaded as part of the project setup; the tarball lives at `data/glodap/`. After extraction, each tracer is a separate NetCDF on a 1° × 1° × 33-depth-level global grid.

Variables used in this notebook:
- **NO3** (μmol/kg) — surface nitrate. Becomes the loss target via the inverse-pattern proxy described in the intro.
- **temperature** (°C) — surface SST. Becomes the single environmental covariate that the DINN conditions Carroll-6 on.

Other GLODAP variables (DIC, ALK, PO4, Si, oxygen, salinity) are present in the dataset and trivially available for follow-up notebooks; not used here to keep the pipeline test minimal.

In [ ]:
glodap_dir = _repo_root / "data" / "glodap" / "GLODAPv2.2016b_MappedClimatologies"
assert glodap_dir.exists(), f"GLODAP directory missing: {glodap_dir}"

no3_full = xr.open_dataset(glodap_dir / "GLODAPv2.2016b.NO3.nc")["NO3"]
temp_full = xr.open_dataset(glodap_dir / "GLODAPv2.2016b.temperature.nc")["temperature"]

# Surface layer.
no3_surf = no3_full.isel(depth_surface=0)
temp_surf = temp_full.isel(depth_surface=0)

print(f"GLODAP global grid: lat {no3_surf.sizes['lat']}, lon {no3_surf.sizes['lon']}")
print(f"NO3 units: {no3_full.attrs.get('units', '?')}, range [{float(no3_surf.min(skipna=True)):.3f}, {float(no3_surf.max(skipna=True)):.3f}]")
print(f"Temperature units: {temp_full.attrs.get('units', 'degC')}, range [{float(temp_surf.min(skipna=True)):.1f}, {float(temp_surf.max(skipna=True)):.1f}]")

## 2. Mid-Atlantic AOI subset

Region: 30–50°N, 60–30°W. ~20°×30° = 600 1°-cells, of which ~500 are open ocean (rest is North America coastline / Europe edge masked as NaN). Chosen to match the AOI decided on the 2026-05-07 collaboration call.

After subsetting, drop any cells where either field is NaN (coastline, missing data) so the optimisation only runs on clean ocean cells. The DINN is per-cell with no spatial coupling, so the irregular mask is fine.

In [ ]:
# GLODAP uses longitude convention 0..360 (specifically 20.5..379.5), not -180..180.
# Mid-Atlantic 60W..30W maps to lon 300..330 in this convention.
midatl_no3 = no3_surf.sel(lat=slice(30, 50), lon=slice(300, 330))
midatl_temp = temp_surf.sel(lat=slice(30, 50), lon=slice(300, 330))

no3_arr = midatl_no3.values   # [H, W]
sst_arr = midatl_temp.values
ocean_mask = ~np.isnan(no3_arr) & ~np.isnan(sst_arr)
print(f"AOI shape: {no3_arr.shape}")
print(f"Ocean cells (non-NaN): {ocean_mask.sum()} of {no3_arr.size}")
print(f"NO3 range in AOI: [{np.nanmin(no3_arr):.3f}, {np.nanmax(no3_arr):.3f}] umol/kg")
print(f"SST range in AOI: [{np.nanmin(sst_arr):.1f}, {np.nanmax(sst_arr):.1f}] degC")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(no3_arr, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title("GLODAP surface NO3 (umol/kg) — Mid-Atl AOI")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(sst_arr, origin="lower", aspect="auto", cmap="plasma")
axes[1].set_title("GLODAP surface temperature (degC) — Mid-Atl AOI")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout(); plt.show()

## 3. Build the per-cell training inputs

Set up tensors of shape `[1, H, W]` for the SST covariate and the NO3 target. The DINN expects `[n_input_channels, H, W]`; we use 1 channel (SST) and produce 6 Carroll-6 outputs per cell.

Normalise SST to zero-mean unit-variance across the AOI ocean cells so the network input is well-conditioned. The NO3 target stays in physical units; the loss handles normalisation.

In [ ]:
# Replace land NaNs with 0 in SST (won't affect loss because mask zeros them out there too).
sst_clean = np.where(ocean_mask, sst_arr, 0.0)
no3_clean = np.where(ocean_mask, no3_arr, 1.0)  # placeholder positive value to keep tensor finite

# Normalise SST over ocean cells only.
sst_ocean_mean = sst_arr[ocean_mask].mean()
sst_ocean_std = sst_arr[ocean_mask].std()
sst_norm = np.where(ocean_mask, (sst_arr - sst_ocean_mean) / sst_ocean_std, 0.0)

env = torch.tensor(sst_norm, dtype=torch.float32).unsqueeze(0)  # [1, H, W]
no3_target = torch.tensor(no3_clean, dtype=torch.float32)         # [H, W]
mask = torch.tensor(ocean_mask, dtype=torch.bool)                  # [H, W]

print(f"env shape: {tuple(env.shape)}, range [{env.min():.2f}, {env.max():.2f}]")
print(f"no3_target shape: {tuple(no3_target.shape)}, range [{no3_target.min():.3f}, {no3_target.max():.3f}]")
print(f"ocean mask: {mask.sum().item()} cells true")

## 4. DINN per-cell setup + carroll6 box model integration

**DINN architecture** (from `src/darwindiff/networks.py`): 1 input channel (SST normalised) → 16 → 16 → 6 outputs, all 1×1 convolutions. ~454 weights.

**Forward pass per epoch:**
1. DINN(SST) → 6 unconstrained Carroll-6 outputs per cell, shape `[6, H, W]`.
2. `bounded_params` maps to physical Carroll-6 ranges via sigmoid.
3. `carroll6_step` integrates the 5-tracer box model 200 forward-Euler steps from a uniform initial state, batched across all H×W cells. The same physics that recovered Carroll's six numbers in notebook 05.
4. Pull the steady-state phyto biomass: $P_s + P_l$ at the final step.
5. Z-score over ocean cells (zero-mean, unit-variance) so the loss compares spatial patterns, not magnitudes.
6. Compute MSE against the pre-computed z-scored target (negative NO$_3$).

**Loss function:** `MSE(z_score(phyto_biomass) - z_score(-NO3))` over ocean cells only. Both fields are zero-mean unit-variance over the AOI ocean cells, so the loss is purely a spatial-pattern-matching loss decoupled from absolute magnitude. The biological intuition: oligotrophic surface waters (low NO$_3$) host higher integrated phytoplankton biomass because nitrate has been drawn down by the standing stock — so the spatial pattern of $P_s + P_l$ from the box model should anti-correlate with the NO$_3$ field, equivalently correlate with $-\text{NO}_3$.

**Why z-scored, not inverse:** an earlier attempt used $1/(\text{NO}_3 + \epsilon)$ as the target. GLODAP's mapped NO$_3$ has small *negative* values in some cells (mapping artifact at very oligotrophic regions, where the gridded mean is effectively zero with sub-grid uncertainty). Inverting near zero produced large outliers and corrupted the loss landscape — the optimiser saturated parameters at bounds. The z-scored formulation is robust to these outliers because it standardises the target distribution before the loss.

In [ ]:
H, W = env.shape[1], env.shape[2]

# Initial state, broadcast to every cell. Same scalar values as notebook 07.
state0_scalar = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025])  # DFe, Ps, Pl, POC, PIC
state0 = state0_scalar.reshape(5, 1, 1).expand(5, H, W).contiguous()

# Integration setup matching notebook 07: 50-day spin-up.
dt = 0.25
n_steps = 200

# Move tensors to the GPU.
env_dev = env.to(device)
state0_dev = state0.to(device)
no3_target_dev = no3_target.to(device)
mask_dev = mask.to(device)
bounds_dev = PARAM_BOUNDS.to(device)

# Build the loss target as z-scored NEGATIVE NO3 over ocean cells.
# Why z-scored, not 1/NO3: GLODAP's mapped NO3 has small negative values in some cells
# (mapping artifact at very oligotrophic regions), so 1/NO3 blows up and corrupts the
# loss. Negative NO3 is a defensible proxy direction (high phyto ↔ low NO3 → -NO3 high)
# and z-scoring kills magnitude/sign issues so the optimisation is conditioned well.
neg_no3 = -no3_target_dev
neg_no3_ocean = neg_no3[mask_dev]
target_mean = neg_no3_ocean.mean()
target_std = neg_no3_ocean.std().clamp(min=1e-6)
target_z = (neg_no3 - target_mean) / target_std
print(f"Pre-computed z-scored -NO3 target. ocean mean={float(target_mean):.3f}, std={float(target_std):.3f}")
print(f"  z-scored target ocean range: [{float(target_z[mask_dev].min()):.2f}, {float(target_z[mask_dev].max()):.2f}]")

## 5. Train DINN to fit the GLODAP NO3 spatial pattern via box-model phyto

Adam at lr=5e-3, 1500 epochs. Same hyperparameters as notebook 07's per-cell fit on synthetic data. Loss is reported every 250 epochs. Forward integration is autograd-backpropagated to update the DINN weights at every epoch.

In [ ]:
torch.manual_seed(0)
dinn = DINN(n_input_channels=1, hidden_dim=16, n_outputs=6).to(device)
optimizer = torch.optim.Adam(dinn.parameters(), lr=5e-3)

n_epochs = 1500
losses: list[float] = []

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
for epoch in range(n_epochs):
    optimizer.zero_grad()
    theta = dinn(env_dev)                        # [6, H, W]
    params = bounded_params(theta, bounds_dev)    # [6, H, W]

    # Forward integrate the box model at every cell.
    state = state0_dev
    for _ in range(n_steps):
        state = carroll6_step(state, params, dt)
    phyto = state[1] + state[2]                   # [H, W], P_s + P_l

    # Z-score the predicted phyto over ocean cells, then MSE against z-scored target.
    # Both fields end up zero-mean unit-variance over ocean cells, so the loss is purely
    # spatial-pattern matching, decoupled from absolute magnitude.
    phyto_ocean = phyto[mask_dev]
    phyto_z = (phyto - phyto_ocean.mean()) / phyto_ocean.std().clamp(min=1e-6)

    residual = (phyto_z - target_z) * mask_dev.to(phyto.dtype)
    loss = (residual ** 2).sum() / mask_dev.sum().to(residual.dtype)

    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 250 == 0:
        print(f"  epoch {epoch + 1:4d}  loss = {loss.item():.4e}")
if device == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - t0
print(f"\nTrained {n_epochs} epochs in {elapsed:.1f}s on {device}")
print(f"Loss: {losses[0]:.3e} -> {losses[-1]:.3e}")

## 6. Recovered Carroll-6 maps + predicted-vs-target pattern comparison

Side-by-side plots of:
- The loss curve (log scale).
- The **inverse-NO3 target** (what we asked the box model's phyto pattern to look like).
- The **recovered phyto pattern** ($P_s + P_l$ from the box model, at integrated steady state, normalised over ocean cells).
- The **per-cell recovered Carroll-6 maps**, all six parameters across the AOI.

Spatial coherence and qualitative match between the inverse-NO3 target and the recovered phyto pattern is the success criterion. The Carroll-6 maps should be smooth functions of SST (since SST is the only DINN input) and physically plausible (within the bounded ranges).

In [ ]:
with torch.no_grad():
    theta_final = dinn(env_dev)
    params_final = bounded_params(theta_final, bounds_dev).cpu()  # [6, H, W]
    state = state0_dev
    for _ in range(n_steps):
        state = carroll6_step(state, bounded_params(theta_final, bounds_dev), dt)
    phyto_final = (state[1] + state[2]).cpu()                       # [H, W]

# Sanity check: the box-model integration must not produce NaN at any ocean
# cell. A single bad cell would silently poison the z-score (mean / std go
# NaN, propagating to the entire field) and corrupt the headline correlation.
assert torch.isfinite(phyto_final[mask]).all(), (
    "phyto_final has non-finite values at ocean cells — "
    "the box-model integration blew up. Inspect params_final for outliers."
)

# Z-score the final predicted phyto over ocean cells for plotting against the z-scored target.
phyto_ocean_final = phyto_final[mask]
phyto_z_final = (phyto_final - phyto_ocean_final.mean()) / phyto_ocean_final.std().clamp(min=1e-6)
target_z_cpu = target_z.cpu()

# Mask land in plots.
phyto_plot = np.where(ocean_mask, phyto_z_final.numpy(), np.nan)
target_plot = np.where(ocean_mask, target_z_cpu.numpy(), np.nan)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].semilogy(losses)
axes[0, 0].set_title("Loss curve (MSE on z-scored fields)")
axes[0, 0].set_xlabel("epoch"); axes[0, 0].grid(alpha=0.3)
axes[0, 1].imshow(np.where(ocean_mask, no3_arr, np.nan), origin="lower", aspect="auto", cmap="viridis")
axes[0, 1].set_title("GLODAP surface NO3 (raw, umol/kg)")
im_target = axes[1, 0].imshow(target_plot, origin="lower", aspect="auto", cmap="plasma")
axes[1, 0].set_title("Loss target: z-scored -NO3")
plt.colorbar(im_target, ax=axes[1, 0])
im_pred = axes[1, 1].imshow(phyto_plot, origin="lower", aspect="auto", cmap="plasma")
axes[1, 1].set_title("DINN-driven box-model phyto biomass (z-scored)")
plt.colorbar(im_pred, ax=axes[1, 1])
plt.tight_layout(); plt.show()

# Pearson correlation between target and prediction over ocean cells.
# Pass RAW fields (not z-scored) so safe_pearson_r's constant-detection
# fires correctly. Pearson r is scale-and-shift invariant, so the value
# is identical when the prediction has real spread; but z-scoring with
# .std().clamp(min=1e-6) magnifies float-level noise from ~1e-15 to ~1e-9,
# moving the spread/|mean| ratio from ~1e-16 to O(1) and defeating the
# relative-tolerance check. Compare raw to raw.
pred_flat = phyto_final.numpy()[ocean_mask]
target_flat = (-no3_target.numpy())[ocean_mask]
result = safe_pearson_r(pred_flat, target_flat)
corr = result.r
print(f"\nPearson correlation (predicted phyto vs -NO3): "
      f"r = {format_pearson(result, n_total=int(ocean_mask.sum()))}")

In [ ]:
# Per-parameter recovered Carroll-6 maps, all six side by side.
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for i, (ax, name) in enumerate(zip(axes.flat, PARAM_NAMES)):
    field = np.where(ocean_mask, params_final[i].numpy(), np.nan)
    im = ax.imshow(field, origin="lower", aspect="auto", cmap="viridis")
    ax.set_title(f"recovered {name}")
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

# Print per-parameter range across the AOI ocean cells.
print("\nRecovered Carroll-6 ranges across Mid-Atl AOI ocean cells:")
for i, name in enumerate(PARAM_NAMES):
    p = params_final[i].numpy()[ocean_mask]
    print(f"  {name:<11s} [{p.min():.4e}, {p.max():.4e}]  mean={p.mean():.4e}")

## 7. Head-to-head — DINN vs Green's-functions parametric class on real data

Notebook 06 demonstrated that ML beats Green's-functions calibration by 15.2× on synthetic two-regime data with planted heterogeneity. The DINN side of that benchmark (covariate-conditioned per-region predictions) won because it could represent regional differences a global-scalar set structurally couldn't.

This section runs the same head-to-head **on the real GLODAP NO3 spatial pattern** in the Mid-Atlantic AOI. Two contestants on the *same* loss target as the DINN above:

1. **Global-scalar fit (Green's-functions parametric class):** six learnable scalars `(alpfe, scav_rat, Smallgrow, Biggrow, diatomgraz, R_PICPOC)`, applied uniformly to *every* cell in the AOI. Same physical bounds via sigmoid. Same forward integration. Same z-scored MSE loss. Same number of epochs. The only difference: the parameter values are a single vector, not a per-cell field.

2. **DINN per-cell fit (DarwinDiff class):** already done above, results stored in `params_final` and `phyto_final`.

The structural prediction: the global-scalar fit can match the AOI mean of the target but cannot reproduce its spatial pattern, because every cell evolves with the same six numbers. Pearson correlation between predicted phyto biomass and z-scored −NO3 should plateau near zero for the global-scalar contestant. DINN already reached r ≈ 0.69 in section 6.

In [ ]:
torch.manual_seed(0)
theta_global = torch.zeros(6, requires_grad=True, device=device)
optimizer_global = torch.optim.Adam([theta_global], lr=5e-2)

losses_global: list[float] = []

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
for epoch in range(n_epochs):
    optimizer_global.zero_grad()
    params_global = bounded_params(theta_global, bounds_dev)  # [6]

    # Forward integrate the box model at every cell, using the SAME global scalar
    # parameters at every cell. Broadcast through carroll6_step naturally — each
    # params[i] is a 0-d tensor and broadcasts against the [H, W] state slices.
    state = state0_dev
    for _ in range(n_steps):
        state = carroll6_step(state, params_global, dt)
    phyto = state[1] + state[2]  # [H, W]

    phyto_ocean = phyto[mask_dev]
    phyto_z = (phyto - phyto_ocean.mean()) / phyto_ocean.std().clamp(min=1e-6)

    residual = (phyto_z - target_z) * mask_dev.to(phyto.dtype)
    loss = (residual ** 2).sum() / mask_dev.sum().to(residual.dtype)

    loss.backward()
    optimizer_global.step()
    losses_global.append(loss.item())
    if (epoch + 1) % 250 == 0:
        print(f"  epoch {epoch + 1:4d}  loss = {loss.item():.4e}")
if device == "cuda":
    torch.cuda.synchronize()
elapsed_global = time.time() - t0
print(f"\nGlobal-scalar fit: {n_epochs} epochs in {elapsed_global:.1f}s on {device}")
print(f"Loss: {losses_global[0]:.3e} -> {losses_global[-1]:.3e}")

with torch.no_grad():
    final_global = bounded_params(theta_global, bounds_dev).cpu()
print("\nRecovered global Carroll-6 (one vector for the entire AOI):")
for i, name in enumerate(PARAM_NAMES):
    print(f"  {name:<11s} = {final_global[i].item():.4e}")

## 8. Comparison and the structural ceiling

Compare the two contestants on:

- **Loss curves** side by side. The global-scalar fit's loss should plateau at a level above DINN's, because it can't represent spatial heterogeneity.
- **Pearson correlation** between predicted phyto biomass and the z-scored −NO3 target. DINN: r ≈ 0.69 (from section 6). Global-scalar prediction: depends on whether the AOI mean alone explains any of the spatial variance.
- **Predicted phyto biomass spatial maps** side by side. The global-scalar prediction will be uniform up to the SST-independent, location-independent dynamics of the box model. DINN's prediction will track the SST gradient.

The structural argument is: *no matter how well-tuned*, a single Carroll-6 vector for the whole ocean cannot reproduce the spatial pattern of nutrient consumption that the DINN's per-cell predictions can. The plateau loss for the global-scalar fit is the ceiling of any approach restricted to one-global-scalar-per-parameter. Carroll's Green's-functions calibration in ECCO-Darwin is exactly that parametric class; this notebook section quantifies the ceiling on real data.

In [ ]:
# Final evaluations for both contestants on the same target.
with torch.no_grad():
    state_g = state0_dev
    for _ in range(n_steps):
        state_g = carroll6_step(state_g, bounded_params(theta_global, bounds_dev), dt)
    phyto_global = (state_g[1] + state_g[2]).cpu()

phyto_g_ocean = phyto_global[mask]
phyto_g_z = (phyto_global - phyto_g_ocean.mean()) / phyto_g_ocean.std().clamp(min=1e-6)
phyto_g_plot = np.where(ocean_mask, phyto_g_z.numpy(), np.nan)

# Pearson correlation for global-scalar fit. Pass RAW fields (not the
# z-scored phyto_g_plot / target_plot) so safe_pearson_r can detect the
# structurally-zero-variance case: z-scoring with .std().clamp(min=1e-6)
# magnifies float-level noise from ~1e-15 to ~1e-9, defeating the
# relative-tolerance constant check. Pearson r is scale-and-shift
# invariant for non-degenerate inputs, so the value is identical when
# the prediction has real spread.
pred_g_flat = phyto_global.numpy()[ocean_mask]
target_flat_raw = (-no3_target.numpy())[ocean_mask]
result_global = safe_pearson_r(pred_g_flat, target_flat_raw)
corr_global = result_global.r  # NaN if is_constant; finite otherwise

print(f"Pearson correlation, predicted phyto vs -NO3 (ocean cells only):")
print(f"  Global-scalar fit (Green's-functions class):  r = {format_pearson(result_global, n_total=int(ocean_mask.sum()))}")
print(f"  DINN per-cell fit (DarwinDiff class):          r = {format_pearson(result, n_total=int(ocean_mask.sum()))}")
print()
print(f"Loss plateau:")
print(f"  Global-scalar: {losses_global[-1]:.4f}")
print(f"  DINN:          {losses[-1]:.4f}")
ratio = losses_global[-1] / max(losses[-1], 1e-12)
print(f"  Loss ratio (Global / DINN): {ratio:.2f}x  (DINN reaches a {ratio:.1f}x lower loss plateau)")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].semilogy(losses_global, label="Global-scalar (Green's class)", color="tab:red")
axes[0, 0].semilogy(losses, label="DINN per-cell", color="tab:green")
axes[0, 0].set_title("Loss curves: same target, two parametric classes")
axes[0, 0].set_xlabel("epoch"); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].imshow(target_plot, origin="lower", aspect="auto", cmap="plasma")
axes[0, 1].set_title("Target: z-scored -NO3 (GLODAP)")

axes[1, 0].imshow(phyto_g_plot, origin="lower", aspect="auto", cmap="plasma")
axes[1, 0].set_title(
    "Global-scalar prediction (" + (
        "undefined" if result_global.is_constant else f"r = {corr_global:.3f}"
    ) + ")"
)

axes[1, 1].imshow(phyto_plot, origin="lower", aspect="auto", cmap="plasma")
axes[1, 1].set_title(
    "DINN prediction (" + (
        "undefined" if result.is_constant else f"r = {corr:.3f}"
    ) + ")"
)

plt.tight_layout(); plt.show()

## What this notebook demonstrates — and what it doesn't

**Demonstrated, on real GLODAP observations (Mid-Atlantic AOI, 582 ocean cells):**

1. **End-to-end data pipeline.** Real ocean BGC NetCDF → xarray → AOI masking → torch tensors → DINN per-cell prediction → carroll6 box-model integration → autograd through the integration → Adam updates the DINN. Every step that worked on synthetic 2-D fields in notebook 07 also works on the real GLODAP grid with land masking.

2. **Loss converges (DINN).** With z-scored target, Adam reduced the loss from 3.00 to **0.617** over 1500 epochs (~5× reduction). The first attempt used $1/\text{NO}_3$ as target and stalled at high loss with parameters saturated at bounds — z-scoring both sides fixed the conditioning.

3. **Recovered Carroll-6 maps are smooth functions of SST.** Because the DINN is a 1×1-conv per-cell network with one input feature, every cell's recovered Carroll-6 vector is a deterministic function of that cell's SST. Recovered ranges across the AOI:

   | Param | Mean | Range |
   |---|---|---|
   | `alpfe` | 0.57 | 0.54–0.58 |
   | `scav_rat` | 2.21e-6 | 1.80e-6 – 2.27e-6 |
   | `Smallgrow` | 0.72 | 0.56–0.78 |
   | `Biggrow` | 1.09 | 0.97–1.15 |
   | `diatomgraz` | 0.58 | 0.50–0.61 |
   | `R_PICPOC` | 0.10 | 0.086–0.103 |

   Recovered values do not match Carroll's published optima (and could not — GLODAP wasn't produced by Darwin). They are the best-fit values for this proxy loss on this real-data spatial pattern.

4. **Spatial-pattern correlation (DINN).** Pearson r = 0.691 (r² = 0.477) between predicted box-model phyto biomass and z-scored −NO3, over 582 ocean cells. The DINN, trained on SST input alone, recovered a parameter-mapping that explains ~ 48 % of the variance in the inverse-nitrate spatial structure.

5. **Head-to-head against Green's-functions parametric class on real data.** The same loss target, the same physics, the same number of epochs — only the parametric class differs:

   | Contestant | Loss plateau | Pearson r | Variance explained |
   |---|---|---|---|
   | Global-scalar (Green's-functions class) | 1.055 | **undefined** (constant prediction) | 0 % |
   | DINN per-cell (DarwinDiff class) | **0.617** | **0.691** | ~ 48 % |

   The global-scalar fit's correlation is undefined because the prediction has zero spatial variance — a single Carroll-6 vector applied uniformly to every cell, evolving from a uniform initial state, produces an identical phyto value at every cell. Pearson r requires non-zero variance on both sides; when the prediction is constant, r is mathematically undefined. *That undefined result is the structural ceiling itself, made visible.* The global-scalar parametric class can't produce a spatial pattern, only a single magnitude — and the loss plateau ≈ variance of the z-scored target (1.0) confirms it: the best a constant prediction can do is "predict the mean for every cell," which explains 0 % of the spatial variance.

   The DINN class subsumes the global-scalar one (it can collapse to a constant by ignoring its input) and reaches a strictly lower loss with a non-zero variance prediction that correlates 0.69 with the target. **This is the same structural argument as notebook 06's 15.2× synthetic win, now demonstrated on real ocean data.**

**Not demonstrated (deferred to notebook 10+):**

- **Carroll-6 recovery against published optima.** GLODAP wasn't produced by Darwin, so the recovered values aren't directly comparable to Carroll's `(0.928, 6e-7, 0.661, 0.431, 0.830, 0.0425)`. The recovered numbers are best-fit-to-this-loss values, not Darwin-equivalent values.
- **Iron-pair identifiability.** GLODAP has no gridded iron field, so `alpfe` and `scav_rat` are unconstrained by this loss. Adding GEOTRACES iron sections (sparse but real) is the next-step deliverable for those two parameters.
- **Multi-tracer joint loss.** Only NO3 is used as the target; DIC, ALK, PO4, Si are all in GLODAP and trivially available. Extending the loss to a multi-tracer composite is straightforward but adds optimisation complexity.
- **Spatial coupling in the box model.** Each cell evolves independently. Real ocean has horizontal advection / diffusion that connects neighbouring cells; that's the next architectural extension.
- **Real Darwin biogeochemistry.** The carroll6 box model is a 5-tracer proxy. Replacing it with the actual MITgcm Darwin pkg (or fitting against ECCO-Darwin v5 output) is what closes the gap from "the methodology runs on real data" to "the methodology recovers Darwin's parameters from real data."

## Where this fits in the project arc

- 05: scalar recovery scaffold (one regime, autodiff matches Carroll's six numbers from synthetic).
- 06: ML vs Green's-functions head-to-head on synthetic two-regime (15× win in per-region recovery error).
- 07: 2-D spatial extension on synthetic; final CPU-vs-GPU benchmark.
- 08: pre-ORCD scoping checkpoint (memory budget, prerequisite checklist).
- **09 (this notebook):** real-data pipeline test on public GLODAP, Mid-Atl AOI; methodology validated end-to-end on real ocean observations (r = 0.69 spatial-pattern match for DINN, undefined / structurally bounded for global-scalar Green's-functions class). The structural argument from 06 now holds on real data.
- 10+: ECCO-Darwin v5 fit on Mid-Atl + Pacific (once Earthdata access is sorted) — the actual Carroll-6-recovery scientific test.
- 11+: real Darwin pkg in the loop and / or vessel observation composite for the iron pair.